# DenseNet121 — Glaucoma Stage Classification

**Pipeline:**
1. Mount Google Drive / clone repo (Colab setup)
2. Load dataset via HuggingFace `datasets`
3. Build DenseNet121 model
4. Train with frozen backbone → unfreeze → fine-tune
5. Evaluate on test set
6. Save metrics + plots

## 0. Colab setup
Run this cell only on **Google Colab**. Skip locally.

In [ ]:
# ── Colab only ────────────────────────────────────────────────────────────────
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # Adjust this path to wherever you cloned the repo in your Drive
    REPO_PATH = '/content/drive/MyDrive/Explainable-Glaucoma-Stage-Classification'
    %cd {REPO_PATH}

    !pip install -q datasets pytorch-grad-cam captum

print('Working directory:', __import__('os').getcwd())

## 1. Imports & reproducibility

In [ ]:
import torch
import torch.nn as nn

from src.training.config import (
    BATCH_SIZE, NUM_EPOCHS, LEARNING_RATE, WEIGHT_DECAY,
    NUM_CLASSES, DEVICE, set_seed,
)
from src.datasets.glaucoma_dataset import build_dataloaders
from src.models.densenet121 import DenseNet121Classifier
from src.training.train_loop import fit, evaluate
from src.utils.metrics import compute_metrics, save_metrics, print_report
from src.utils.visualize import plot_training_curves, plot_confusion_matrix

set_seed()
print(f'Device: {DEVICE}')

## 2. Load data

In [ ]:
train_loader, val_loader, test_loader = build_dataloaders(batch_size=BATCH_SIZE)

print(f'Train batches : {len(train_loader):>4}  ({len(train_loader.dataset)} images)')
print(f'Val   batches : {len(val_loader):>4}  ({len(val_loader.dataset)} images)')
print(f'Test  batches : {len(test_loader):>4}  ({len(test_loader.dataset)} images)')

## 3. Build model

In [ ]:
model = DenseNet121Classifier(
    num_classes=NUM_CLASSES,
    pretrained=True,
    freeze_features=True,   # frozen backbone for Phase 1
).to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable:,} / {total:,}')

## 4. Phase 1 — Train classifier head only (frozen backbone)

Start with a frozen backbone so the randomly-initialised head converges quickly
without corrupting the pretrained features.

In [ ]:
PHASE1_EPOCHS = 5

loss_fn   = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE * 10,   # higher LR is fine while backbone is frozen
    weight_decay=WEIGHT_DECAY,
)

history_phase1 = fit(
    model, train_loader, val_loader,
    loss_fn, optimizer,
    num_epochs=PHASE1_EPOCHS,
    model_name='densenet121',
    patience=3,
)

## 5. Phase 2 — Fine-tune full network

Unfreeze all layers and continue with a much smaller learning rate.

In [ ]:
model.unfreeze_features()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable params after unfreeze: {trainable:,}')

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,        # 1e-4 — small LR to preserve pretrained weights
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS, eta_min=1e-6
)

history_phase2 = fit(
    model, train_loader, val_loader,
    loss_fn, optimizer,
    num_epochs=NUM_EPOCHS,
    model_name='densenet121',
    patience=5,
)

## 6. Training curves

In [ ]:
# Merge both phases for a single continuous plot
full_history = {
    'train_loss': history_phase1['train_loss'] + history_phase2['train_loss'],
    'val_loss':   history_phase1['val_loss']   + history_phase2['val_loss'],
}

fig = plot_training_curves(full_history, model_name='densenet121')
fig.show()

## 7. Load best checkpoint & evaluate on test set

In [ ]:
from src.training.config import CHECKPOINT_DIR
import os

best_ckpt = os.path.join(CHECKPOINT_DIR, 'densenet121_best.pt')
model.load_state_dict(torch.load(best_ckpt, map_location=DEVICE))
print(f'Loaded best checkpoint from {best_ckpt}')

test_loss, test_preds, test_labels = evaluate(model, test_loader, loss_fn)
print(f'Test loss: {test_loss:.4f}')

## 8. Metrics

In [ ]:
metrics = compute_metrics(test_labels, test_preds)

print(f"Accuracy : {metrics['accuracy']:.4f}")
print(f"F1 macro : {metrics['f1_macro']:.4f}")
print()
print_report(test_labels, test_preds)

save_metrics(metrics, model_name='densenet121')

## 9. Confusion matrix

In [ ]:
fig_cm = plot_confusion_matrix(metrics['confusion_matrix'], model_name='densenet121')
fig_cm.show()